## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Capability contract

**Capability:** C03 — Correctness Contracts & Idempotency  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Detect a correctness failure


## 🖊️ Sketch note

![Sketch note — Data Quality](../assets/sketch-notes/12-data-quality.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Your task

**Goal:** Turn data-quality assumptions into an explicit publication policy.

**Do this:** Add amount-range and future-date checks, classify each as block/quarantine/observe, and justify the policy with evidence.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S03 — Data quality (DATA-1063)** → [`../work_simulations/03_acme_commerce_quality_gate/README.md`](../work_simulations/03_acme_commerce_quality_gate/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Learning objectives

- write checks for completeness, uniqueness and validity;
- distinguish a failed check from a failed pipeline;
- measure quality results;
- decide what should be quarantined.


## 1. Mental model

```text
data artifact
     ↓
quality checks ──→ evidence
     ↓                ↓
 quality result   failures / metrics
     ↓
 publish? ── yes / no
```

Quality is multidimensional. A dataset can be complete but still invalid, unique but stale, or structurally valid but referentially broken.

## 2. Predict before you run

A table has 100 rows. One row has a null customer ID, two rows share the same order ID, and every value is syntactically valid.

> Which quality dimensions fail? Which ones would you make publication-blocking?

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"duckdb": "duckdb>=1.0,<2", "pyarrow": "pyarrow>=16,<22", "pandas": "pandas>=2,<4", "requests": "requests>=2.31,<3"}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import duckdb

WORK = Path("data/12_data_quality")
WORK.mkdir(parents=True, exist_ok=True)

customers = pd.DataFrame([
    {"customer_id": "C1", "name": "Ada"},
    {"customer_id": "C2", "name": "Lin"},
    {"customer_id": "C3", "name": "Sam"},
])
orders = pd.DataFrame([
    {"order_id": "O100", "customer_id": "C1", "amount": 120.0, "status": "paid", "event_time": datetime.now(timezone.utc) - timedelta(minutes=5)},
    {"order_id": "O101", "customer_id": "C2", "amount": 80.0, "status": "paid", "event_time": datetime.now(timezone.utc) - timedelta(minutes=4)},
    {"order_id": "O102", "customer_id": "C3", "amount": 50.0, "status": "pending", "event_time": datetime.now(timezone.utc) - timedelta(minutes=3)},
])

con = duckdb.connect()
con.register("customers", customers)
con.register("orders", orders)
print(orders)

## 3. Build reusable checks

Each check returns a small, machine-readable result. This is more useful operationally than printing a warning and moving on.

In [ ]:
def result(name, passed, observed, expected, details=""):
    return {
        "check": name,
        "passed": bool(passed),
        "observed": observed,
        "expected": expected,
        "details": details,
    }

def not_null(df, column):
    bad = int(df[column].isna().sum())
    return result(f"not_null:{column}", bad == 0, bad, 0)

def unique(df, column):
    duplicates = int(df[column].duplicated().sum())
    return result(f"unique:{column}", duplicates == 0, duplicates, 0)

def allowed_values(df, column, allowed):
    invalid = sorted(set(df[column].dropna()) - set(allowed))
    return result(f"allowed_values:{column}", not invalid, len(invalid), 0, str(invalid))

def positive(df, column):
    bad = int((df[column] <= 0).sum())
    return result(f"positive:{column}", bad == 0, bad, 0)

def foreign_key(child, child_col, parent, parent_col):
    parent_values = set(parent[parent_col].dropna())
    missing = int((~child[child_col].isin(parent_values)).sum())
    return result(f"foreign_key:{child_col}->{parent_col}", missing == 0, missing, 0)

def freshness(df, timestamp_col, max_age_minutes):
    now = datetime.now(timezone.utc)
    age = (now - pd.to_datetime(df[timestamp_col], utc=True)).dt.total_seconds() / 60
    stale = int((age > max_age_minutes).sum())
    return result(f"freshness:{timestamp_col}", stale == 0, stale, 0, f"max age={age.max():.1f} min")

## 4. Run the quality suite

In [ ]:
checks = []
checks += [not_null(orders, "order_id"), not_null(orders, "customer_id")]
checks += [unique(orders, "order_id")]
checks += [positive(orders, "amount")]
checks += [allowed_values(orders, "status", {"pending", "paid", "cancelled"})]
checks += [foreign_key(orders, "customer_id", customers, "customer_id")]
checks += [freshness(orders, "event_time", max_age_minutes=30)]

quality = pd.DataFrame(checks)
display(quality)

publishable = bool(quality["passed"].all())
print("\nPublishable:", publishable)
assert publishable

## 5. Measure quality, not just pass/fail

A useful quality report preserves evidence: what was checked, what was observed, what was expected and whether publication is allowed.

In [ ]:
print("Checks:", len(quality))
print("Passed:", int(quality["passed"].sum()))
print("Failed:", int((~quality["passed"]).sum()))
quality.to_json(WORK / "quality_result.json", orient="records", indent=2)
print("Quality result written to", WORK / "quality_result.json")

## 💥 6. Break it deliberately

Introduce three realistic failures: a duplicate key, an unknown customer and a stale event.

In [ ]:
broken = orders.copy()
broken.loc[1, "order_id"] = broken.loc[0, "order_id"]
broken.loc[2, "customer_id"] = "C999"
broken.loc[0, "event_time"] = datetime.now(timezone.utc) - timedelta(hours=2)

broken_checks = []
broken_checks += [not_null(broken, "customer_id"), unique(broken, "order_id")]
broken_checks += [foreign_key(broken, "customer_id", customers, "customer_id")]
broken_checks += [freshness(broken, "event_time", 30)]
broken_quality = pd.DataFrame(broken_checks)
display(broken_quality)
assert not broken_quality["passed"].all()

## 7. Diagnose → Improve

The quality report localizes the failure to specific dimensions instead of forcing an operator to inspect the entire dataset manually.

The smallest improvement is not “clean everything.” It is to **block publication for checks that protect downstream correctness**, while deciding explicitly which anomalies may be quarantined or monitored.

In [ ]:
blocking_checks = {"unique:order_id", "foreign_key:customer_id->customer_id", "not_null:customer_id"}
failed_blockers = broken_quality[
    (~broken_quality["passed"]) & broken_quality["check"].isin(blocking_checks)
]
print("Blocking failures:")
display(failed_blockers)
print("Publication decision:", "BLOCK" if not failed_blockers.empty else "ALLOW")

## 8. What changes in production?

At scale, quality becomes a first-class platform capability: contracts are versioned, checks run automatically, results are stored over time, alerts have owners, and quarantined records can be replayed.

Not every check should be blocking. A freshness breach may trigger an incident for one product but only a warning for another. The policy is part of the data product contract.

## 9. Explain it in 60–90 seconds

> Why is “data quality” not one metric? Give an example where a dataset passes completeness but is still unsafe to publish.

## 10. Challenge

Add two checks:
- numeric `amount` must be within a business range;
- no order may be dated in the future.

Then classify each check as **block**, **quarantine**, or **observe** and justify the decision.

## Takeaways

- Quality assumptions should be executable.
- A quality result is evidence plus a publication decision.
- Completeness, uniqueness, validity, integrity and freshness catch different failure modes.
- Blocking policy is a design decision, not an automatic consequence of a failed check.

---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 12](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/12.md)
